# Training FNOs on Darcy-Flow with TensorGRaD

This notebook follows the same basic workflow as the NeuralOperator Darcy FNO example: load Darcy-Flow data, build an FNO, set up losses and optimizers, train, and visualize results.

We compare three optimizers:

- AdamW baseline;
- TensorGRaD low-rank gradients with a 25% budget;
- TensorGRaD low-rank + sparse gradients with a matched 20% + 5% budget.

## Import dependencies

The path setup below makes sure this notebook uses the local NeuralOperator checkout when run from the repository.


In [ ]:
from pathlib import Path
import sys


for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "neuralop" / "__init__.py").exists():
        repo_root = path
        break
else:
    repo_root = None

if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"NeuralOperator root: {repo_root if repo_root is not None else 'installed package'}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from neuralop import H1Loss, LpLoss, Trainer
from neuralop.data.datasets.darcy import DarcyDataset
from neuralop.models import FNO
from neuralop.training import AdamW as BaselineAdamW, TensorGRaD, fno_tensorgrad_param_groups
from neuralop.utils import count_model_params


plt.rcParams["figure.dpi"] = 110

## Configuration

In [ ]:
N_TRAIN = 1000
N_TEST = 100
BATCH_SIZE = 128
N_EPOCHS = 15
LEARNING_RATE = 1e-2
WEIGHT_DECAY = 1e-4
HIDDEN_CHANNELS = 24
N_MODES = (8, 8)
UPDATE_PROJ_GAP = 4
LOW_RANK_SCALE = 10.0
SPARSE_SCALE = 1
LAMBDA_SPARSE = 1
SPARSE_TYPE = "randk"

LOW_RANK_BUDGET = 0.25
LOW_RANK_SPARSE_LOW_RANK_BUDGET = 0.20
LOW_RANK_SPARSE_SPARSE_BUDGET = 0.05
assert abs(LOW_RANK_BUDGET - (LOW_RANK_SPARSE_LOW_RANK_BUDGET + LOW_RANK_SPARSE_SPARSE_BUDGET)) < 1e-12

DATA_ROOT = Path.home() / ".cache" / "neuraloperator" / "darcy"
TEST_RESOLUTIONS = [16, 32]
TEST_BATCH_SIZES = [BATCH_SIZE, BATCH_SIZE]

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Loading the Darcy-Flow dataset

`DarcyDataset(download=True)` downloads the public Darcy files from Zenodo if they are not already present in `DATA_ROOT`. This keeps the example independent of any local repository data.


In [ ]:
darcy_dataset = DarcyDataset(
    root_dir=DATA_ROOT,
    n_train=N_TRAIN,
    n_tests=[N_TEST, N_TEST],
    batch_size=BATCH_SIZE,
    test_batch_sizes=TEST_BATCH_SIZES,
    train_resolution=16,
    test_resolutions=TEST_RESOLUTIONS,
    encode_input=False,
    encode_output=True,
    download=True,
)

train_loader = DataLoader(
    darcy_dataset.train_db,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loaders = {
    resolution: DataLoader(
        darcy_dataset.test_dbs[resolution],
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    for resolution, batch_size in zip(TEST_RESOLUTIONS, TEST_BATCH_SIZES)
}
data_processor = darcy_dataset.data_processor.to(device)
print(f"Loaded Darcy-Flow test resolutions: {list(test_loaders)}")


## Creating the FNO model

In [ ]:
def make_model():
    torch.manual_seed(0)
    return FNO(
        n_modes=N_MODES,
        in_channels=1,
        out_channels=1,
        hidden_channels=HIDDEN_CHANNELS,
        projection_channel_ratio=2,
    ).to(device)


preview_model = make_model()
print(f"FNO parameters: {count_model_params(preview_model):,}")
del preview_model

## Creating the optimizers

TensorGRaD is applied to the FNO spectral convolution weights. Other parameters, such as lifting/projection layers and skip connections, stay in a normal AdamW group.

The compressed-gradient budgets are matched:

- low-rank only: 25%;
- low-rank + sparse residual: 20% low-rank + 5% sparse.


In [ ]:
def adamw_optimizer(model):
    return BaselineAdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )


def tensorgrad_low_rank_optimizer(model):
    param_groups = fno_tensorgrad_param_groups(
        model,
        rank=LOW_RANK_BUDGET,
        min_params=1000,
        update_proj_gap=UPDATE_PROJ_GAP,
        scale=LOW_RANK_SCALE,
    )
    return TensorGRaD(param_groups, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def tensorgrad_low_rank_sparse_optimizer(model):
    param_groups = fno_tensorgrad_param_groups(
        model,
        rank=LOW_RANK_SPARSE_LOW_RANK_BUDGET,
        sparse_ratio=LOW_RANK_SPARSE_SPARSE_BUDGET,
        min_params=1000,
        update_proj_gap=UPDATE_PROJ_GAP,
        scale=LOW_RANK_SCALE,
        sparse_scale=SPARSE_SCALE,
        sparse_type=SPARSE_TYPE,
        lambda_sparse=LAMBDA_SPARSE,
    )
    return TensorGRaD(param_groups, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


print(f"Low-rank budget: {LOW_RANK_BUDGET:.0%}")
print(
    "Low-rank + sparse budget: "
    f"{LOW_RANK_SPARSE_LOW_RANK_BUDGET:.0%} + "
    f"{LOW_RANK_SPARSE_SPARSE_BUDGET:.0%} = "
    f"{LOW_RANK_BUDGET:.0%}"
)


## Setting up losses

In [ ]:
l2loss = LpLoss(d=2, p=2)
h1loss = H1Loss(d=2)

train_loss = h1loss
eval_losses = {"h1": h1loss, "l2": l2loss}

## Training all three models

In [ ]:
def to_float(value):
    if torch.is_tensor(value):
        return float(value.detach().cpu())
    if value is None:
        return None
    return float(value)


class HistoryTrainer(Trainer):
    def train_one_epoch(self, epoch, train_loader, training_loss):
        train_err, avg_loss, avg_lasso_loss, epoch_train_time = super().train_one_epoch(
            epoch,
            train_loader,
            training_loss,
        )
        self._latest_history_row = {
            "epoch": epoch,
            "step": (epoch + 1) * len(train_loader),
            "train_err": to_float(train_err),
            "avg_loss": to_float(avg_loss),
            "epoch_train_time": to_float(epoch_train_time),
        }
        return train_err, avg_loss, avg_lasso_loss, epoch_train_time

    def evaluate_all(
        self,
        epoch,
        eval_losses,
        test_loaders,
        eval_modes,
        max_autoregressive_steps=None,
    ):
        eval_metrics = super().evaluate_all(
            epoch=epoch,
            eval_losses=eval_losses,
            test_loaders=test_loaders,
            eval_modes=eval_modes,
            max_autoregressive_steps=max_autoregressive_steps,
        )
        row = dict(getattr(self, "_latest_history_row", {"epoch": epoch}))
        row.update({key: to_float(value) for key, value in eval_metrics.items()})
        self.history.append(row)
        return eval_metrics


def train_with_optimizer(name, optimizer_builder):
    torch.manual_seed(0)
    np.random.seed(0)
    model = make_model()
    optimizer = optimizer_builder(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    print(f"\n### {name} ###")
    print(f"Optimizer: {optimizer.__class__.__name__}")

    trainer = HistoryTrainer(
        model=model,
        n_epochs=N_EPOCHS,
        device=device,
        data_processor=data_processor,
        wandb_log=False,
        eval_interval=1,
        use_distributed=False,
        verbose=True,
    )
    trainer.history = []
    trainer.train(
        train_loader=train_loader,
        test_loaders=test_loaders,
        optimizer=optimizer,
        scheduler=scheduler,
        regularizer=False,
        training_loss=train_loss,
        eval_losses=eval_losses,
    )
    return model, trainer.history


optimizer_builders = {
    "AdamW": adamw_optimizer,
    "Low-rank 25%": tensorgrad_low_rank_optimizer,
    "Low-rank 20% + sparse 5%": tensorgrad_low_rank_sparse_optimizer,
}

models = {}
histories = {}
for name, optimizer_builder in optimizer_builders.items():
    models[name], histories[name] = train_with_optimizer(name, optimizer_builder)


## Loss curves

All panels use optimizer steps on the x-axis. Validation H1 and L2 are shown at the 32 x 32 test resolution.

In [ ]:
plot_resolution = max(test_loaders)
eval_h1_key = f"{plot_resolution}_h1"
eval_l2_key = f"{plot_resolution}_l2"

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, history in histories.items():
    steps = [row["step"] for row in history]
    axes[0].plot(steps, [row["avg_loss"] for row in history], marker="o", label=name)
    axes[1].plot(steps, [row[eval_h1_key] for row in history], marker="o", label=name)
    axes[2].plot(steps, [row[eval_l2_key] for row in history], marker="o", label=name)

axes[0].set_title("Train H1 loss")
axes[1].set_title(f"Validation H1 loss ({plot_resolution} x {plot_resolution})")
axes[2].set_title(f"Validation L2 loss ({plot_resolution} x {plot_resolution})")

for ax in axes:
    ax.set_xlabel("Optimizer step")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## Visualizing predictions

The columns compare the input, ground truth, AdamW prediction, low-rank prediction, and low-rank + sparse prediction at the 32 x 32 test resolution.

In [ ]:
plot_resolution = max(test_loaders)
test_samples = test_loaders[plot_resolution].dataset
prediction_labels = ["AdamW", "Low-rank 25%", "Low-rank 20% + sparse 5%"]

fig = plt.figure(figsize=(12, 7))
for index in range(3):
    data = test_samples[index]
    data = data_processor.preprocess(data, batched=False)

    x = data["x"].to(device)
    y = data["y"].to(device)

    with torch.no_grad():
        predictions = {
            label: models[label](x.unsqueeze(0)).squeeze().detach().cpu()
            for label in prediction_labels
        }

    panels = [
        ("Input x", x[0].detach().cpu(), "gray"),
        ("Ground truth y", y.squeeze().detach().cpu(), None),
        *[(label, predictions[label], None) for label in prediction_labels],
    ]
    for column, (title, image, cmap) in enumerate(panels):
        ax = fig.add_subplot(3, 5, index * 5 + column + 1)
        ax.imshow(image, cmap=cmap)
        if index == 0:
            ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle(f"Darcy-Flow FNO predictions at {plot_resolution} x {plot_resolution}", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Optional hyperparameter sweep.
# AdamW has no projector gap, so it is run once per learning rate.
# TensorGRaD variants are run for every (learning rate, projector gap, scale) tuple.

SWEEP_LEARNING_RATES = [1e-2, 3e-2]
SWEEP_PROJECTOR_GAPS = [8]
SWEEP_SCALES = [10, 5, 1]

plot_resolution = max(test_loaders)
eval_h1_key = f"{plot_resolution}_h1"
eval_l2_key = f"{plot_resolution}_l2"


def best_metric(history, key):
    best_row = min(history, key=lambda row: row[key])
    return best_row[key], best_row["step"]


def set_projector_gap(projector_gap):
    global UPDATE_PROJ_GAP
    UPDATE_PROJ_GAP = projector_gap


def set_projector_scale(scale):
    global LOW_RANK_SCALE
    LOW_RANK_SCALE = scale


def run_sweep_case(method_name, optimizer_builder, learning_rate, projector_gap=None, scale=None):
    global LEARNING_RATE

    LEARNING_RATE = learning_rate
    if projector_gap is not None:
        set_projector_gap(projector_gap)
    if scale is not None:
        set_projector_scale(scale)

    run_name = f"{method_name} | lr={learning_rate:g}"
    if projector_gap is not None:
        run_name += f" | gap={projector_gap}"
    if scale is not None:
        run_name += f" | scale={scale:g}"

    model, history = train_with_optimizer(run_name, optimizer_builder)
    best_h1, best_h1_step = best_metric(history, eval_h1_key)
    best_l2, best_l2_step = best_metric(history, eval_l2_key)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "method": method_name,
        "learning_rate": learning_rate,
        "projector_gap": projector_gap,
        "scale": scale,
        "best_val_h1": best_h1,
        "best_val_h1_step": best_h1_step,
        "best_val_l2": best_l2,
        "best_val_l2_step": best_l2_step,
    }


def print_sweep_results(results):
    header = (
        f"{'method':<28} {'lr':>10} {'gap':>6} {'scale':>8} "
        f"{'best_h1':>12} {'h1_step':>8} {'best_l2':>12} {'l2_step':>8}"
    )
    print(header)
    print("-" * len(header))
    for row in results:
        gap = "-" if row["projector_gap"] is None else str(row["projector_gap"])
        scale = "-" if row["scale"] is None else f"{row['scale']:g}"
        print(
            f"{row['method']:<28} "
            f"{row['learning_rate']:>10.2e} "
            f"{gap:>6} "
            f"{scale:>8} "
            f"{row['best_val_h1']:>12.5g} "
            f"{row['best_val_h1_step']:>8} "
            f"{row['best_val_l2']:>12.5g} "
            f"{row['best_val_l2_step']:>8}"
        )


base_learning_rate = LEARNING_RATE
base_projector_gap = UPDATE_PROJ_GAP
base_projector_scale = LOW_RANK_SCALE
sweep_results = []
try:
    for learning_rate in SWEEP_LEARNING_RATES:
        sweep_results.append(
            run_sweep_case("AdamW", adamw_optimizer, learning_rate)
        )

        for projector_gap in SWEEP_PROJECTOR_GAPS:
            for scale in SWEEP_SCALES:
                sweep_results.append(
                    run_sweep_case(
                        "Low-rank 25%",
                        tensorgrad_low_rank_optimizer,
                        learning_rate,
                        projector_gap,
                        scale,
                    )
                )
                sweep_results.append(
                    run_sweep_case(
                        "Low-rank 20% + sparse 5%",
                        tensorgrad_low_rank_sparse_optimizer,
                        learning_rate,
                        projector_gap,
                        scale,
                    )
                )
finally:
    LEARNING_RATE = base_learning_rate
    set_projector_gap(base_projector_gap)
    set_projector_scale(base_projector_scale)

sweep_results = sorted(
    sweep_results,
    key=lambda row: (
        row["method"],
        row["learning_rate"],
        -1 if row["projector_gap"] is None else row["projector_gap"],
        -1 if row["scale"] is None else row["scale"],
    ),
)
print_sweep_results(sweep_results)
